# MeaningGuard: Colab (free T4) pipeline
Run the cells top to bottom. Runtime -> Change runtime type -> T4 GPU. Replace `GITHUB_URL` in cell 2 (placeholder).

In [ ]:
# (1) Mount Drive (checkpoints are copied here so a disconnect does not lose them)
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted")

In [ ]:
# (2) Clone repo (PLACEHOLDER URL: replace with your GitHub URL)
GITHUB_URL = "https://github.com/<YOUR-USERNAME>/meaningguard.git"
!git clone $GITHUB_URL /content/meaningguard
%cd /content/meaningguard
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# (3) Install dependencies
!pip install -q -r requirements.txt
print("install done")

In [ ]:
# (4) Data pipeline. USE_HF=False reproduces the committed dataset (built-in 300 seeds) that final_report.md describes.
# USE_HF=True tries LingoIITGN/COMI-LINGUA first: the dataset then changes, so regenerate every table afterwards.
USE_HF = False
flag = "" if USE_HF else "--force_fallback"
!python scripts/extract_seeds.py $flag
!python scripts/generate_variants.py
!python scripts/split_by_seed.py
!python scripts/validate_dataset.py

In [ ]:
# (5) Train main model (MuRIL-base, multi-task). Prints peak GPU memory each epoch.
!python -m meaningguard.train --config configs/muril_base.yaml
!mkdir -p /content/drive/MyDrive/meaningguard_ckpt && cp -r checkpoints/muril_base /content/drive/MyDrive/meaningguard_ckpt/
print("main model done")

In [ ]:
# (6) All baselines (Qwen 4-bit is the slowest: ~420 generations)
!python baselines/dummy.py
!python baselines/zeroshot_encoder.py
!python baselines/translation_pipeline.py
!python baselines/zeroshot_llm.py
print("baselines done")

In [ ]:
# (7) Ablations: A1 + the three A3 encoders (one after another; each frees GPU memory on exit)
for cfg in ["a1_binary_only", "a3_base_swap_muril", "a3_base_swap_indicbert", "a3_base_swap_xlmr"]:
    print("=== running", cfg)
    !python -m meaningguard.train --config ablation/{cfg}.yaml
!cp -r checkpoints /content/drive/MyDrive/meaningguard_ckpt/all_runs
print("ablations done")

In [ ]:
# (8) Evaluate everything + generate tables + error taxonomy dump
!python -m meaningguard.evaluate --all
!python -m meaningguard.evaluate --errors predictions/muril_base_test.jsonl --gold data/processed/test.jsonl
!python -m meaningguard.evaluate --all
!cat reports/results_table.md reports/ablation_table.md
!cp -r reports predictions /content/drive/MyDrive/meaningguard_ckpt/